In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install mediapipe

In [ ]:
!pip install mediapipe --upgrade

In [ ]:
!pip install mediapipe==0.10.21

In [4]:
import os
import cv2
import mediapipe as mp
import pandas as pd
import scipy.io as sio
import numpy as np

# 1. Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True, 
    max_num_faces=1, 
    refine_landmarks=False, 
    min_detection_confidence=0.5
)

# 2. Define Dataset Paths
# Kaggle mounts your datasets here
dataset_path = '/kaggle/input/datasets/maulidio16/300w-lp/300W_LP'  
output_csv = '/kaggle/working/head_pose_training_data.csv'

# Let's start by just processing one of the sub-folders to test speed
sub_folder = 'AFW' 
folder_path = os.path.join(dataset_path, sub_folder)

data_rows = []
print("Starting extraction... This might take a minute.")

# 3. Loop through the images
for filename in os.listdir(folder_path):
    if filename.endswith(".jpg"):
        base_name = filename.replace('.jpg', '')
        img_path = os.path.join(folder_path, filename)
        mat_path = os.path.join(folder_path, base_name + '.mat')
        
        # Check if the annotation file exists
        if not os.path.exists(mat_path):
            continue
            
        # Load the image
        image = cv2.imread(img_path)
        if image is None: continue
        
        # Convert BGR to RGB for MediaPipe
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Process with MediaPipe
        results = face_mesh.process(image_rgb)
        
        if results.multi_face_landmarks:
            # Extract the ground truth angles (Pitch, Yaw, Roll) from the .mat file
            mat_data = sio.loadmat(mat_path)
            # The 'Pose_Para' array contains the angles in the first 3 indices
            pose = mat_data['Pose_Para'][0][:3] 
            pitch, yaw, roll = pose[0], pose[1], pose[2]
            
            # Extract the 468 X, Y landmarks
            face_landmarks = results.multi_face_landmarks[0]
            landmarks = []
            for landmark in face_landmarks.landmark:
                landmarks.extend([landmark.x, landmark.y])
            
            # Combine landmarks and target angles into one row
            row = landmarks + [pitch, yaw, roll]
            data_rows.append(row)
            
        # Optional: break early just to test the script (remove this later)
       
# 4. Save to CSV
# Create column names: x0, y0, x1, y1... pitch, yaw, roll
columns = []
for i in range(468):
    columns.extend([f'x{i}', f'y{i}'])
columns.extend(['pitch', 'yaw', 'roll'])

df = pd.DataFrame(data_rows, columns=columns)
df.to_csv(output_csv, index=False)

print(f"Data saved to {output_csv}")
print(df.head())

Starting extraction... This might take a minute.


W0000 00:00:1775196111.064963     315 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775196111.077819     316 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Data saved to /kaggle/working/head_pose_training_data.csv
         x0        y0        x1        y1        x2        y2        x3  \
0  0.321663  0.729714  0.302654  0.686521  0.317980  0.701793  0.286286   
1  0.668154  0.685580  0.698946  0.626108  0.665588  0.649451  0.666042   
2  0.676762  0.705621  0.704201  0.651962  0.669157  0.672377  0.671125   
3  0.685910  0.699934  0.716089  0.650200  0.677852  0.667425  0.683767   
4  0.687332  0.705620  0.705366  0.642704  0.675798  0.664534  0.662254   

         y3        x4        y4  ...      y464      x465      y465      x466  \
0  0.639265  0.298044  0.670389  ...  0.595726  0.335961  0.599501  0.434648   
1  0.574797  0.700865  0.608518  ...  0.545442  0.651759  0.547800  0.682067   
2  0.609291  0.705666  0.637067  ...  0.563774  0.641388  0.568615  0.645470   
3  0.602521  0.717716  0.635356  ...  0.560375  0.657130  0.564631  0.662748   
4  0.597299  0.704380  0.626410  ...  0.550049  0.642210  0.554132  0.669930   

       y46

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

print("🚀 Initializing Head Pose Training Pipeline...")

# DUAL GPU CONFIGURATION
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Detected GPUs: {torch.cuda.device_count()}")
if torch.cuda.device_count() > 0:
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Configuration - OPTIMIZED FOR DUAL T4
CONFIG = {
    'data_path': '/kaggle/working/head_pose_training_data.csv',
    'model_output': '/kaggle/working/head_pose_model.pth',
    'scaler_output': '/kaggle/working/head_pose_scaler.pkl',
    'batch_size': 256,      # Doubled for dual GPU
    'epochs': 100,
    'patience': 15,
    'lr': 0.001,
    'weight_decay': 1e-4
}

print("\n📊 Step 1-3: Loading & Preprocessing Data...")

if not os.path.exists(CONFIG['data_path']):
    raise FileNotFoundError(f"Dataset not found at {CONFIG['data_path']}")

df = pd.read_csv(CONFIG['data_path'])
print(f"Loaded {len(df)} samples")

initial_count = len(df)
df = df.dropna()
print(f"Dropped {initial_count - len(df)} rows with missing values")

if len(df) == 0:
    raise ValueError("No data remaining after dropping NaNs")

X = df.iloc[:, :-3].values
y = df.iloc[:, -3:].values

print(f"Feature matrix shape: {X.shape}")
print(f"Target matrix shape: {y.shape}")

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, CONFIG['scaler_output'])
print(f"✅ Scaler saved to {CONFIG['scaler_output']}")

train_dataset = TensorDataset(
    torch.FloatTensor(X_train_scaled), 
    torch.FloatTensor(y_train)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val_scaled), 
    torch.FloatTensor(y_val)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_scaled), 
    torch.FloatTensor(y_test)
)

# DataLoader workers optimized for dual GPU
num_workers = 4 if torch.cuda.is_available() else 0
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], 
                          shuffle=True, pin_memory=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], 
                        shuffle=False, pin_memory=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], 
                         shuffle=False, pin_memory=True, num_workers=num_workers)

print("\n🧠 Step 4: Building Enhanced Neural Network...")

class HeadPoseNet(nn.Module):
    def __init__(self, input_dim=936):
        super(HeadPoseNet, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, 3)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.network(x)

model = HeadPoseNet()

# WRAP MODEL FOR DUAL GPU (if available)
if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = nn.DataParallel(model)
    
model = model.to(device)

print("\n⚙️ Step 5: Setting up Loss, Optimizer & Scheduler...")

criterion = nn.SmoothL1Loss(beta=1.0)
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

# 🔥 FIXED: Removed verbose=True (not supported in older PyTorch versions)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print("\n🏋️ Step 6: Training with Early Stopping...")

best_val_loss = float('inf')
best_model_state = None
history = {'train_loss': [], 'val_loss': [], 'val_mae': []}
early_stop_counter = 0

for epoch in range(CONFIG['epochs']):
    # Training Phase
    model.train()
    train_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_mae = 0.0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            
            val_loss += criterion(outputs, targets).item()
            mae = torch.abs(outputs - targets).mean(dim=0)
            val_mae += mae.sum().item()
    
    avg_val_loss = val_loss / len(val_loader)
    avg_val_mae = val_mae / (len(val_loader) * 3)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_mae'].append(avg_val_mae)
    
    # Get current LR before step
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    
    # 🔥 Manual LR reduction logging (since verbose=True removed)
    if new_lr < current_lr:
        print(f"🔽 LR reduced: {current_lr:.6f} -> {new_lr:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # SAVE MODULE STATE (unwrap DataParallel if needed)
        if isinstance(model, nn.DataParallel):
            best_model_state = model.module.state_dict().copy()
        else:
            best_model_state = model.state_dict().copy()
        early_stop_counter = 0
        print(f"💾 New best model saved! (Val Loss: {avg_val_loss:.4f})")
    else:
        early_stop_counter += 1
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{CONFIG['epochs']}] "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val MAE: {avg_val_mae:.4f}° | "
              f"LR: {current_lr:.6f}")
    
    if early_stop_counter >= CONFIG['patience']:
        print(f"\n⏹️ Early stopping triggered at epoch {epoch+1}")
        break

print("\n📈 Step 7: Final Evaluation on Test Set...")

# Load best model
if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(best_model_state)
else:
    model.load_state_dict(best_model_state)

model.eval()
test_loss = 0.0
test_mae_per_angle = torch.zeros(3).to(device)
all_predictions = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        
        test_loss += criterion(outputs, targets).item()
        mae_per_batch = torch.abs(outputs - targets).mean(dim=0)
        test_mae_per_angle += mae_per_batch
        
        all_predictions.append(outputs.cpu())
        all_targets.append(targets.cpu())

avg_test_loss = test_loss / len(test_loader)
test_mae_per_angle = test_mae_per_angle / len(test_loader)

all_predictions = torch.cat(all_predictions, dim=0)
all_targets = torch.cat(all_targets, dim=0)

angle_names = ['Pitch', 'Yaw', 'Roll']
print("\n🎯 Test Set Performance:")
print(f"Overall MSE Loss: {avg_test_loss:.4f}")
print("Per-Angle MAE (degrees):")
for i, name in enumerate(angle_names):
    mae = torch.abs(all_predictions[:, i] - all_targets[:, i]).mean().item()
    std = torch.abs(all_predictions[:, i] - all_targets[:, i]).std().item()
    print(f"  {name}: {mae:.2f}° (±{std:.2f}°)")

print("\n💾 Step 8: Exporting Production Model...")

# SAVE UNWRAPPED STATE_DICT (compatible with single-GPU inference)
torch.save({
    'model_state_dict': best_model_state,
    'model_architecture': 'HeadPoseNet',
    'input_dim': 936,
    'output_dim': 3,
    'best_val_loss': best_val_loss,
    'config': CONFIG,
    'scaler_path': CONFIG['scaler_output']
}, CONFIG['model_output'])

print(f"✅ Model saved to: {CONFIG['model_output']}")
print(f"✅ Scaler saved to: {CONFIG['scaler_output']}")
print(f"✅ Best Validation Loss: {best_val_loss:.4f}")

print("\n🎉 Training Complete! Ready for Flask integration.")
print("\n⚠️  IMPORTANT: Flask deployment uses SINGLE GPU/CPU:")
print("   - The saved .pth works on any device (no DataParallel wrapper)")
print("   - Load normally: model.load_state_dict(torch.load('model.pth')['model_state_dict'])")

🚀 Initializing Head Pose Training Pipeline...
Detected GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4

📊 Step 1-3: Loading & Preprocessing Data...
Loaded 4286 samples
Dropped 0 rows with missing values
Feature matrix shape: (4286, 936)
Target matrix shape: (4286, 3)
Train: 3001, Val: 642, Test: 643
✅ Scaler saved to /kaggle/working/head_pose_scaler.pkl

🧠 Step 4: Building Enhanced Neural Network...
🚀 Using 2 GPUs with DataParallel

⚙️ Step 5: Setting up Loss, Optimizer & Scheduler...

🏋️ Step 6: Training with Early Stopping...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return F.linear(input, self.weight, self.bias)


💾 New best model saved! (Val Loss: 0.2179)
Epoch [001/100] Train Loss: 0.2189 | Val Loss: 0.2179 | Val MAE: 0.4577° | LR: 0.001000
💾 New best model saved! (Val Loss: 0.0812)
💾 New best model saved! (Val Loss: 0.0505)
💾 New best model saved! (Val Loss: 0.0279)
Epoch [005/100] Train Loss: 0.0608 | Val Loss: 0.0400 | Val MAE: 0.1562° | LR: 0.001000
💾 New best model saved! (Val Loss: 0.0261)
💾 New best model saved! (Val Loss: 0.0226)
💾 New best model saved! (Val Loss: 0.0223)
Epoch [010/100] Train Loss: 0.0452 | Val Loss: 0.0234 | Val MAE: 0.1287° | LR: 0.001000
💾 New best model saved! (Val Loss: 0.0221)
💾 New best model saved! (Val Loss: 0.0202)
Epoch [015/100] Train Loss: 0.0349 | Val Loss: 0.0231 | Val MAE: 0.1212° | LR: 0.001000
💾 New best model saved! (Val Loss: 0.0186)
💾 New best model saved! (Val Loss: 0.0173)
Epoch [020/100] Train Loss: 0.0329 | Val Loss: 0.0178 | Val MAE: 0.1084° | LR: 0.001000
🔽 LR reduced: 0.001000 -> 0.000500
Epoch [025/100] Train Loss: 0.0326 | Val Loss: 0.019